In [ ]:
"""
OOMMF Simulation Analysis — Multi-file Overlay + SFD
=====================================================
Loads one or more CONVERTED simulation CSVs, asks for sphere diameter and
cap thickness per file, and produces two outputs:

  1. Overlay plot (one figure, all files together):
       Top panel    : physical magnetic moment [A·m²]  — Moment_Am2 column
       Bottom panel : normalised magnetisation M/Msat  — Mz/Ms column
     Each file is a labelled dashed coloured line; useful for comparing:
       - same simulation parameters, different diameters
       - same diameter, different simulation parameters

  2. Per-file individual plot (hysteresis + SFD below):
       Top panel    : Mz/Ms hysteresis loop
       Bottom panel : Switching field distribution |dM/dH| of descending branch
     + a plain-text .txt report with Hc, Mr/Ms, SFD FWHM per file.

Expected CSV columns (CONVERTED format):
  B_ext (T), Mz/Ms, Moment_Am2

Inputs  : One or more CONVERTED simulation CSVs (selected via GUI dialog).
          Sphere diameter [µm] and cap thickness [nm] entered per file.
Outputs : Overlay PNG + SVG, per-file PNG + SVG + .txt report — all saved to
          a timestamped folder inside `ubermag_outputs/`.

# ─────────────────────────────────────────────────────────────────────────────
# Copyright (C) 2026 Max-Planck-Gesellschaft zur Förderung der Wissenschaften e.V.
#               2026 University of Stuttgart
# Authors: Natalia Gonzalez-Vazquez, Andrew K. Schulz
# SPDX-License-Identifier: GPL-3.0-or-later
#
# This file is part of FunMaP.
# FunMaP is free software: you can redistribute it and/or modify it under
# the terms of the GNU General Public License as published by the Free
# Software Foundation, either version 3 of the License, or (at your option)
# any later version.
#
# FunMaP is distributed in the hope that it will be useful, but WITHOUT ANY
# WARRANTY; without even the implied warranty of MERCHANTABILITY or FITNESS
# FOR A PARTICULAR PURPOSE. See the GNU General Public License for more details.
#
# You should have received a copy of the GNU General Public License along with
# FunMaP. If not, see <https://www.gnu.org/licenses/>.
# ─────────────────────────────────────────────────────────────────────────────
"""

# --- Standard library ---
import os
import re
from datetime import datetime

# --- Third-party ---
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --- GUI ---
import tkinter as tk
from tkinter import filedialog, simpledialog


# ===========================================================================
# 1. SETTINGS
# ===========================================================================

OUTDIR_BASE       = "ubermag_outputs"
N_GRID            = 2500   # Interpolation grid points for SFD
STD_SMOOTH_WINDOW = 71     # Moving-average window for SFD smoothing (odd)

# Keywords for auto-detecting the field column (fallback for non-CONVERTED files)
FIELD_KEYWORDS = ['B_ext (T)', 'B_ext', 'Field', 'H_ext', 'B_hysteresis']
MAG_NORM_COL   = 'Mz/Ms'       # Normalised magnetisation column
MAG_PHYS_COL   = 'Moment_Am2'  # Physical moment column [A·m²]


# ===========================================================================
# 2. UTILITY FUNCTIONS
# ===========================================================================

def moving_average(y, n):
    """Box-car moving average of window n (forced odd). Returns y if n < 3."""
    if n < 3:
        return y
    if n % 2 == 0:
        n += 1
    return np.convolve(np.asarray(y), np.ones(n) / n, mode="same")


def calculate_fwhm(h, dm_dh):
    """
    Estimate the full-width at half-maximum of the SFD peak.
    Returns NaN if fewer than two points exceed the half-maximum threshold.
    """
    peak_val = np.max(dm_dh)
    half_max = peak_val / 2.0
    indices  = np.where(dm_dh >= half_max)[0]
    if len(indices) < 2:
        return np.nan
    return abs(h[indices[-1]] - h[indices[0]])


def safe_coercivity(h, m):
    """
    Estimate coercivity by linear interpolation across the M = 0 crossing.
    Returns NaN if the branch does not cross zero (incomplete loop).
    """
    h, m     = np.asarray(h), np.asarray(m)
    sort_idx = np.argsort(m)
    m_s, h_s = m[sort_idx], h[sort_idx]
    if m_s[0] > 0 or m_s[-1] < 0:
        return np.nan
    return np.interp(0, m_s, h_s)


def split_branches(H, m):
    """
    Split a full hysteresis loop into descending (H+ → H−) and ascending
    (H− → H+) branches based on the positions of the field extrema.
    """
    idx_max, idx_min = np.argmax(H), np.argmin(H)
    if idx_max < idx_min:
        h_desc, m_desc = H[idx_max:idx_min+1], m[idx_max:idx_min+1]
        h_asc  = np.concatenate([H[idx_min:], H[:idx_max]])
        m_asc  = np.concatenate([m[idx_min:], m[:idx_max]])
    else:
        h_asc,  m_asc  = H[idx_min:idx_max+1], m[idx_min:idx_max+1]
        h_desc = np.concatenate([H[idx_max:], H[:idx_min]])
        m_desc = np.concatenate([m[idx_max:], m[:idx_min]])
    return (h_desc, m_desc), (h_asc, m_asc)


def find_field_col(df_cols):
    """Auto-detect the field column by matching against known keywords."""
    return next(
        (c for c in df_cols if any(k in c for k in FIELD_KEYWORDS)),
        df_cols[0]
    )


# ===========================================================================
# 3. MAIN
# ===========================================================================

def main():

    # --- File selection ---
    root = tk.Tk()
    root.withdraw()
    root.attributes("-topmost", True)
    paths = filedialog.askopenfilenames(
        title="Select CONVERTED simulation CSV files",
        filetypes=[("CSV files", "*.csv"), ("ODT files", "*.odt")]
    )
    root.destroy()
    if not paths:
        return

    # --- Per-file metadata: diameter and thickness ---
    # Entered interactively so the same script handles any combination of
    # diameters / parameters without needing to encode them in filenames.
    print("\nFor each file, enter the sphere diameter and cap thickness.")
    file_meta = []
    for p in paths:
        fname = os.path.basename(p)
        print(f"\n  File: {fname}")
        diam  = float(input("    Sphere diameter (µm): ").strip())
        thick = float(input("    Cap thickness (nm)  : ").strip())
        file_meta.append({"path": p, "diam": diam, "thick": thick,
                          "label": f"{diam} µm / {thick:.0f} nm"})

    # --- Output folder ---
    ts     = datetime.now().strftime("%Y%m%d_%H%M%S")
    outdir = os.path.join(OUTDIR_BASE, f"analysis_{ts}")
    os.makedirs(outdir, exist_ok=True)

    # --- Parse all files ---
    scans = []
    for meta in file_meta:
        p = meta["path"]
        try:
            if p.endswith('.csv'):
                df = pd.read_csv(p)
            else:
                # ODT: skip comment lines, read column names from "# Columns:" header
                with open(p, 'r') as f:
                    lines = f.readlines()
                cols = []
                for line in lines:
                    if line.startswith("# Columns:"):
                        cols = re.findall(r'\{[^\}]+\}|[^\s{}]+',
                                          line.replace("# Columns:", "").strip())
                        cols = [c.strip('{}') for c in cols]
                        break
                data_start = next(i for i, l in enumerate(lines) if not l.startswith("#"))
                df = pd.read_csv(p, sep=r'\s+', skiprows=data_start,
                                 names=cols, engine='python')

            h_col = find_field_col(df.columns)
            h     = df[h_col].values
            m_norm  = df[MAG_NORM_COL].values  if MAG_NORM_COL  in df.columns else None
            m_phys  = df[MAG_PHYS_COL].values  if MAG_PHYS_COL  in df.columns else None

            if m_norm is None:
                print(f"[!] '{MAG_NORM_COL}' not found in {os.path.basename(p)} — skipping.")
                continue

            # Split into branches for metrics; use sweep order for plotting
            # (preserves the correct loop shape with the switching step)
            (h_desc, m_desc), (h_asc, m_asc) = split_branches(h, m_norm)
            hc1    = safe_coercivity(h_desc, m_desc)
            hc2    = safe_coercivity(h_asc,  m_asc)
            hc_avg = (abs(hc1) + abs(hc2)) / 2
            mr_ms  = (
                np.interp(0, h_desc[::-1], m_desc[::-1])
                + abs(np.interp(0, h_asc, m_asc))
            ) / 2

            scans.append({
                "label":    meta["label"],
                "base_name": os.path.splitext(os.path.basename(p))[0],
                "h":        h,
                "m_norm":   m_norm,
                "m_phys":   m_phys,
                "h_label":  h_col,
                "branches": ((h_desc, m_desc), (h_asc, m_asc)),
                "Hc":       hc_avg,
                "Mr_over_Ms": mr_ms,
            })

        except Exception as e:
            print(f"[!] Error reading {os.path.basename(p)}: {e}")

    if not scans:
        print("[!] No files loaded successfully.")
        return

    colors = plt.rcParams['axes.prop_cycle'].by_key()['color']

    # =========================================================================
    # OUTPUT 1: Overlay plot — all files on one two-panel figure
    # =========================================================================
    fig_ov, (ax_ov1, ax_ov2) = plt.subplots(2, 1, figsize=(9, 11))

    has_phys = any(s["m_phys"] is not None for s in scans)

    for i, s in enumerate(scans):
        color = colors[i % len(colors)]

        # Top panel: physical moment [A·m²] (if available)
        if s["m_phys"] is not None:
            ax_ov1.plot(s["h"], s["m_phys"],
                        lw=1.8, alpha=0.85, color=color, label=s["label"])

        # Bottom panel: normalised Mz/Ms
        ax_ov2.plot(s["h"], s["m_norm"],
                    lw=1.8, alpha=0.85, color=color, label=s["label"])

    ax_ov1.set_ylabel(r"$m$ (A·m²)", fontsize=13)
    ax_ov1.set_title("Simulation comparison — physical moment", fontweight='bold')
    if not has_phys:
        ax_ov1.text(0.5, 0.5, "No Moment_Am2 column found",
                    transform=ax_ov1.transAxes, ha='center', va='center',
                    color='grey', fontsize=11)

    ax_ov2.set_ylabel(r"$M/M_{\mathrm{sat}}$", fontsize=13)
    ax_ov2.set_xlabel(r"$\mu_0 H$ (T)", fontsize=13)
    ax_ov2.set_title("Simulation comparison — normalised M/Msat", fontweight='bold')

    for ax in [ax_ov1, ax_ov2]:
        ax.axhline(0, color='black', lw=0.5)
        ax.axvline(0, color='black', lw=0.5)
        ax.grid(True, ls=':', alpha=0.6)
        ax.legend(loc='best', fontsize='small', frameon=True)

    fig_ov.tight_layout()
    ov_base = os.path.join(outdir, f"Overlay_{ts}")
    fig_ov.savefig(ov_base + ".png", dpi=300)
    fig_ov.savefig(ov_base + ".svg")
    plt.close(fig_ov)
    print(f"\nOverlay plot saved: {ov_base}.png / .svg")

    # =========================================================================
    # OUTPUT 2: Per-file individual plot (hysteresis + SFD) + report
    # =========================================================================
    h_grid = np.linspace(
        min(s["h"].min() for s in scans),
        max(s["h"].max() for s in scans),
        N_GRID
    )

    for i, s in enumerate(scans):
        output_base = os.path.join(outdir, f"{s['base_name']}_{ts}")

        # SFD from descending branch interpolated onto common h_grid
        (hd, md), _ = s["branches"]
        m_interp = np.interp(h_grid, hd[np.argsort(hd)], md[np.argsort(hd)])
        dm_dh    = np.gradient(m_interp, h_grid)
        sfd_fwhm = calculate_fwhm(h_grid, np.abs(dm_dh))

        fig, (ax1, ax2) = plt.subplots(
            2, 1, figsize=(7.5, 9),
            gridspec_kw={'height_ratios': [2, 1]}
        )

        # Top: full Mz/Ms hysteresis loop in original sweep order
        ax1.plot(s["h"], s["m_norm"], color='tab:blue', lw=2, label=s["label"])
        ax1.set_ylabel(r"$M/M_{\mathrm{sat}}$")
        ax1.set_title(f"Simulation: {s['label']}\n{s['base_name']}", fontweight='bold')
        ax1.axhline(0, color='black', lw=0.5)
        ax1.axvline(0, color='black', lw=0.5)
        ax1.grid(True, ls='--', alpha=0.4)
        ax1.legend(fontsize=8)

        # Bottom: SFD |dM/dH| of descending branch
        ax2.plot(h_grid, np.abs(dm_dh), color='tab:red', lw=1.5,
                 label="|dM/dH| (descending)")
        ax2.set_xlabel(r"$\mu_0 H$ (T)")
        ax2.set_ylabel("Susceptibility (arb. u.)")
        ax2.axhline(0, color='black', lw=0.5)
        ax2.grid(True, ls='--', alpha=0.4)
        ax2.legend(fontsize=8)

        fig.tight_layout()
        fig.savefig(output_base + ".png", dpi=300)
        fig.savefig(output_base + ".svg", format='svg')
        plt.close(fig)

        # Plain-text report
        with open(output_base + "_report.txt", "w") as f:
            f.write("Simulation Analysis Report\n")
            f.write(f"File      : {s['base_name']}\n")
            f.write(f"Label     : {s['label']}\n")
            f.write(f"Timestamp : {ts}\n")
            f.write("-" * 40 + "\n")
            f.write(f"Coercivity (Hc)    : {s['Hc']:.6f} T\n")
            f.write(f"Squareness (Mr/Ms) : {s['Mr_over_Ms']:.6f}\n")
            f.write(f"SFD FWHM           : {sfd_fwhm:.6f} T\n")

    print(f"Per-file plots and reports saved in: {outdir}")
    print(f"\nDONE. All outputs in: {outdir}")


if __name__ == "__main__":
    main()
